# Efficient Attention and Inference — Theory Notebook

Interactive implementations of inference mathematics for LLMs.

**Topics covered:**

1. **Roofline model** — arithmetic intensity and the bandwidth wall
2. **KV cache sizing** — memory formulas for multi-head, GQA, MQA
3. **Online softmax** — the mathematical foundation of FlashAttention
4. **FlashAttention IO analysis** — HBM access comparison
5. **PagedAttention** — fragmentation analysis and memory utilisation
6. **Speculative decoding** — acceptance probability and expected speedup
7. **Quantisation error** — scale, zero-point, and per-group quantisation
8. **SmoothQuant** — migrating quantisation difficulty
9. **Batch size vs arithmetic intensity** — finding the optimal batch
10. **Decode speed limit** — bandwidth-bound token generation rates

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def print_table(headers, rows, col_widths=None):
    """Pretty-print a table with headers and rows."""
    if col_widths is None:
        col_widths = [max(len(str(h)), max(len(str(r[i])) for r in rows))
                      for i, h in enumerate(headers)]
    header_str = ' | '.join(str(h).ljust(w) for h, w in zip(headers, col_widths))
    print(header_str)
    print('-' * len(header_str))
    for row in rows:
        print(' | '.join(str(v).ljust(w) for v, w in zip(row, col_widths)))

def fmt_sci(x, digits=2):
    """Format number in scientific notation."""
    if x == 0:
        return '0'
    exp = int(np.floor(np.log10(abs(x))))
    mantissa = x / 10**exp
    if abs(exp) <= 2:
        return f'{x:.{digits}f}'
    return f'{mantissa:.{digits}f}e{exp}'

def fmt_params(n):
    """Format parameter count (e.g., 7.0B, 140M)."""
    if n >= 1e12:
        return f'{n/1e12:.1f}T'
    elif n >= 1e9:
        return f'{n/1e9:.1f}B'
    elif n >= 1e6:
        return f'{n/1e6:.0f}M'
    elif n >= 1e3:
        return f'{n/1e3:.0f}K'
    return str(int(n))

def fmt_bytes(b):
    """Format byte count (e.g., 1.07 GB, 140 MB)."""
    if b >= 1e12:
        return f'{b/1e12:.2f} TB'
    elif b >= 1e9:
        return f'{b/1e9:.2f} GB'
    elif b >= 1e6:
        return f'{b/1e6:.1f} MB'
    elif b >= 1e3:
        return f'{b/1e3:.1f} KB'
    return f'{b:.0f} B'

print("Setup complete ✓")

## §1 Roofline Model

In [ ]:
# === §1: Roofline Model ===
np.random.seed(42)

def roofline_throughput(intensity, peak_flops, peak_bandwidth):
    """Compute achievable throughput from roofline model.
    
    Args:
        intensity: arithmetic intensity (FLOPs/byte)
        peak_flops: peak compute (FLOPS)
        peak_bandwidth: peak memory bandwidth (bytes/sec)
    Returns:
        throughput: achievable FLOPS
    """
    return min(peak_flops, peak_bandwidth * intensity)

def arithmetic_intensity_decode(seq_len, d_k, n_heads, batch_size=1, bytes_per_elem=2):
    """Compute arithmetic intensity for decode attention.
    
    Args:
        seq_len: current sequence length (KV cache entries)
        d_k: head dimension
        n_heads: number of attention heads
        batch_size: batch size
        bytes_per_elem: bytes per weight/activation element
    Returns:
        intensity: FLOPs/byte
    """
    # FLOPs: Q*K^T (s*d_k per head) + softmax*V (s*d_k per head), per head
    # Total: 2 * s * d_k * n_heads * batch_size
    flops = 2 * seq_len * d_k * n_heads * batch_size
    # Bytes: load KV cache (2 * s * d_k * n_heads) + query (d_k * n_heads)
    bytes_accessed = (2 * seq_len * d_k * n_heads + d_k * n_heads) * bytes_per_elem
    return flops / bytes_accessed

# --- 1a: GPU specifications ---
print("=== §1: Roofline Model ===\n")

gpus = {
    'A100 SXM': {'flops': 312e12, 'bw': 2.0e12, 'hbm_gb': 80},
    'H100 SXM': {'flops': 989e12, 'bw': 3.35e12, 'hbm_gb': 80},
    'H200 SXM': {'flops': 989e12, 'bw': 4.8e12, 'hbm_gb': 141},
}

print("Ridge points (I* = Peak FLOPS / Peak Bandwidth):")
rows = []
for name, spec in gpus.items():
    ridge = spec['flops'] / spec['bw']
    rows.append([name, f"{spec['flops']/1e12:.0f} TFLOPS",
                 f"{spec['bw']/1e12:.1f} TB/s", f"{ridge:.0f} FLOPs/byte"])
print_table(['GPU', 'Peak BF16', 'HBM BW', 'Ridge I*'], rows)

# --- 1b: Decode attention intensity ---
print("\n--- Decode Attention Arithmetic Intensity ---")
rows = []
for s in [512, 2048, 8192, 32768, 131072]:
    for B in [1, 8, 64]:
        I = arithmetic_intensity_decode(s, d_k=128, n_heads=32, batch_size=B)
        regime = 'compute' if I > 156 else 'BANDWIDTH'
        rows.append([f"{s:>6}", f"{B:>5}", f"{I:>8.2f}", regime])
print_table(['SeqLen', 'Batch', 'Intensity', 'Regime (A100)'], rows)

# --- 1c: Roofline visualisation ---
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    intensities = np.logspace(-2, 4, 500)
    
    for name, spec in gpus.items():
        throughputs = [roofline_throughput(I, spec['flops'], spec['bw']) for I in intensities]
        ax.plot(intensities, np.array(throughputs)/1e12, linewidth=2, label=name)
    
    # Mark decode attention points
    for s, marker in [(2048, 'o'), (8192, 's'), (32768, '^')]:
        I_val = arithmetic_intensity_decode(s, 128, 32, batch_size=1)
        t_val = roofline_throughput(I_val, 312e12, 2e12)
        ax.scatter([I_val], [t_val/1e12], s=100, marker=marker, zorder=5,
                   label=f'Decode B=1, s={s}')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Arithmetic Intensity (FLOPs/byte)', fontsize=12)
    ax.set_ylabel('Throughput (TFLOPS)', fontsize=12)
    ax.set_title('Roofline Model — GPU Inference', fontsize=14)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('inference_roofline.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: inference_roofline.png")

## §2 KV Cache Sizing

In [ ]:
# === §2: KV Cache Sizing ===
np.random.seed(42)

def kv_cache_bytes(n_layers, n_kv_heads, d_k, seq_len, bytes_per_elem=2):
    """Compute KV cache size in bytes.
    
    Args:
        n_layers: number of transformer layers
        n_kv_heads: number of KV heads (= n_heads for MHA, < n_heads for GQA)
        d_k: head dimension
        seq_len: sequence length
        bytes_per_elem: 2 for BF16, 1 for INT8, 0.5 for INT4
    Returns:
        size in bytes
    """
    return 2 * n_layers * n_kv_heads * d_k * seq_len * bytes_per_elem

# --- 2a: Model configurations ---
print("=== §2: KV Cache Sizing ===\n")

models = {
    'LLaMA-3 8B':  {'L': 32, 'H': 32, 'H_kv': 8,  'd_k': 128, 'params': 8e9},
    'LLaMA-3 70B': {'L': 80, 'H': 64, 'H_kv': 8,  'd_k': 128, 'params': 70e9},
    'Mistral 7B':  {'L': 32, 'H': 32, 'H_kv': 8,  'd_k': 128, 'params': 7e9},
    'GPT-4 (est)': {'L': 120,'H': 96, 'H_kv': 96, 'd_k': 128, 'params': 1.7e12},
}

seq_lens = [4096, 8192, 32768, 131072]

print("KV Cache Size (BF16):")
headers = ['Model'] + [f's={s//1024}K' for s in seq_lens]
rows = []
for name, cfg in models.items():
    row = [name]
    for s in seq_lens:
        size = kv_cache_bytes(cfg['L'], cfg['H_kv'], cfg['d_k'], s, bytes_per_elem=2)
        row.append(fmt_bytes(size))
    rows.append(row)
print_table(headers, rows)

# --- 2b: GQA vs MHA savings ---
print("\n--- GQA vs MHA KV Cache (LLaMA-3 70B, s=128K, BF16) ---")
cfg = models['LLaMA-3 70B']
for label, h_kv in [('MHA (H_kv=64)', 64), ('GQA (H_kv=8)', 8), ('MQA (H_kv=1)', 1)]:
    size = kv_cache_bytes(cfg['L'], h_kv, cfg['d_k'], 131072, 2)
    print(f"  {label:20s}: {fmt_bytes(size):>10}")

# --- 2c: Quantisation impact ---
print("\n--- KV Cache with Quantisation (LLaMA-3 8B, s=8192) ---")
cfg = models['LLaMA-3 8B']
for label, bpe in [('BF16', 2), ('INT8', 1), ('INT4', 0.5)]:
    size = kv_cache_bytes(cfg['L'], cfg['H_kv'], cfg['d_k'], 8192, bpe)
    print(f"  {label:8s}: {fmt_bytes(size):>10}")

# --- 2d: Concurrent requests ---
print("\n--- Concurrent Requests on 80 GB GPU ---")
gpu_mem = 80e9
for name, cfg in [('LLaMA-3 8B', models['LLaMA-3 8B']),
                   ('LLaMA-3 70B', models['LLaMA-3 70B'])]:
    weight_mem = cfg['params'] * 2  # BF16
    available = gpu_mem - weight_mem
    for s in [4096, 8192, 32768]:
        kv_per_req = kv_cache_bytes(cfg['L'], cfg['H_kv'], cfg['d_k'], s, 2)
        n_reqs = int(available / kv_per_req)
        print(f"  {name}, s={s:>5}: {n_reqs:>4} concurrent requests "
              f"(KV/req = {fmt_bytes(kv_per_req)}, avail = {fmt_bytes(available)})")

## §3 Online Softmax

In [ ]:
# === §3: Online Softmax ===
np.random.seed(42)

def standard_softmax(scores):
    """Standard two-pass softmax."""
    m = np.max(scores)
    e = np.exp(scores - m)
    return e / np.sum(e)

def online_softmax_attention(score_blocks, value_blocks):
    """Online softmax with incremental rescaling (FlashAttention core).
    
    Args:
        score_blocks: list of 1D arrays (attention scores per block)
        value_blocks: list of 2D arrays (values per block, shape [block_size, d_v])
    Returns:
        output: weighted sum of values using exact softmax
    """
    d_v = value_blocks[0].shape[1] if value_blocks[0].ndim > 1 else 1
    
    m = -np.inf          # running max
    l = 0.0              # running sum of exp
    o = np.zeros(d_v)    # running output
    
    for scores_b, values_b in zip(score_blocks, value_blocks):
        # New block max
        m_new = max(m, np.max(scores_b))
        
        # Rescale previous accumulator
        rescale = np.exp(m - m_new)
        
        # New exp weights for this block
        exp_b = np.exp(scores_b - m_new)
        
        # Update running sum
        l_new = l * rescale + np.sum(exp_b)
        
        # Update running output
        o = (l * rescale * o + exp_b @ values_b) / l_new
        
        # Update state
        m = m_new
        l = l_new
    
    return o

# --- 3a: Verify online = standard ---
print("=== §3: Online Softmax ===\n")

# Create random scores and values
n_tokens = 12
d_v = 4
scores_full = np.random.randn(n_tokens) * 3  # varied scores
values_full = np.random.randn(n_tokens, d_v)

# Standard attention
weights_std = standard_softmax(scores_full)
output_std = weights_std @ values_full

# Online attention in 3 blocks of 4
block_size = 4
score_blocks = [scores_full[i:i+block_size] for i in range(0, n_tokens, block_size)]
value_blocks = [values_full[i:i+block_size] for i in range(0, n_tokens, block_size)]
output_online = online_softmax_attention(score_blocks, value_blocks)

print(f"Standard softmax output:  [{', '.join(f'{v:.6f}' for v in output_std)}]")
print(f"Online softmax output:   [{', '.join(f'{v:.6f}' for v in output_online)}]")
print(f"Max absolute error:      {np.max(np.abs(output_std - output_online)):.2e}")
print(f"Match: {np.allclose(output_std, output_online)}")

# --- 3b: Step-by-step trace ---
print("\n--- Step-by-Step Online Softmax Trace ---")
m = -np.inf
l = 0.0
print(f"{'Block':>6} {'Scores':>25} {'m_old':>8} {'m_new':>8} {'l_old':>10} {'l_new':>10}")
print("-" * 75)
for i, (sb, vb) in enumerate(zip(score_blocks, value_blocks)):
    m_new = max(m, np.max(sb))
    rescale = np.exp(m - m_new) if m != -np.inf else 0.0
    exp_b = np.exp(sb - m_new)
    l_new = l * rescale + np.sum(exp_b)
    scores_str = '[' + ', '.join(f'{s:.1f}' for s in sb) + ']'
    print(f"{i:>6} {scores_str:>25} {m:>8.2f} {m_new:>8.2f} {l:>10.4f} {l_new:>10.4f}")
    m = m_new
    l = l_new

# --- 3c: Numerical stability demonstration ---
print("\n--- Numerical Stability ---")
large_scores = np.array([1000.0, 1001.0, 999.0])
values_3 = np.eye(3)

# Naive softmax (no max subtraction) — would overflow
try:
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        naive = np.exp(large_scores)
        naive_norm = naive / np.sum(naive)
    print(f"Naive softmax on [1000, 1001, 999]: {naive_norm}  (may show nan/inf)")
except:
    print("Naive softmax overflows!")

# Online softmax handles it correctly
online_result = online_softmax_attention([large_scores], [values_3])
stable_result = standard_softmax(large_scores) @ values_3
print(f"Online softmax:           [{', '.join(f'{v:.6f}' for v in online_result)}]")
print(f"Standard (max-shifted):   [{', '.join(f'{v:.6f}' for v in stable_result)}]")
print(f"Both handle large values correctly via max subtraction.")

## §4 FlashAttention IO Analysis

In [ ]:
# === §4: FlashAttention IO Analysis ===
np.random.seed(42)

def standard_attention_io(n, d, bytes_per_elem=2):
    """HBM IO for standard attention (bytes).
    
    Reads: Q, K (2*n*d each for score), S (n*n), V (n*d)
    Writes: S (n*n), P (n*n after softmax), O (n*d)
    """
    # Read Q, K for scores: 2*n*d; write S: n*n
    # Read S for softmax: n*n; write P: n*n
    # Read P, V for output: n*n + n*d; write O: n*d
    reads = 2*n*d + n*n + n*n + n*d  # Q,K + S + P,V
    writes = n*n + n*n + n*d         # S + P + O
    return (reads + writes) * bytes_per_elem

def flash_attention_io(n, d, M, bytes_per_elem=2):
    """HBM IO for FlashAttention (bytes).
    
    Reads Q, K, V once each (total 3*n*d)
    Writes O once (n*d)
    Inner loop tile reads: O(n^2 * d / M)
    """
    # Read Q, K, V: 3*n*d; write O: n*d
    # Plus tiling reads: O(n^2 * d / M) for re-reading Q blocks
    base_io = 4 * n * d  # Q, K, V read + O write
    tile_io = n * n * d / M  # extra reads from tiling
    return (base_io + tile_io) * bytes_per_elem

# --- 4a: IO comparison across sequence lengths ---
print("=== §4: FlashAttention IO Analysis ===\n")

d = 128          # head dimension
M_sram = 100e3   # 100 KB SRAM (in elements)

print("HBM IO Comparison (per head, BF16):")
rows = []
for n in [512, 1024, 2048, 4096, 8192, 16384]:
    io_std = standard_attention_io(n, d)
    io_fa = flash_attention_io(n, d, M_sram)
    ratio = io_std / io_fa
    rows.append([f"{n:>6}", fmt_bytes(io_std), fmt_bytes(io_fa), f"{ratio:.2f}×"])
print_table(['n', 'Standard IO', 'FlashAttn IO', 'Speedup'], rows)

# --- 4b: Memory footprint ---
print("\n--- HBM Memory Footprint ---")
rows = []
for n in [2048, 8192, 32768, 131072]:
    std_mem = n * n * 2     # S + P matrices in BF16
    fa_mem = 0              # No intermediate matrices
    rows.append([f"{n:>6}", fmt_bytes(std_mem), '0 (tiled in SRAM)', f"{std_mem/1e6:.1f} MB saved"])
print_table(['n', 'Standard', 'FlashAttention', 'Savings'], rows)

# --- 4c: Multi-head total IO ---
print("\n--- Full Model IO (32 heads, BF16) ---")
H = 32
rows = []
for n in [2048, 8192, 32768]:
    io_std = standard_attention_io(n, d) * H
    io_fa = flash_attention_io(n, d, M_sram) * H
    # Time at 2 TB/s
    t_std = io_std / 2e12 * 1000  # ms
    t_fa = io_fa / 2e12 * 1000
    rows.append([f"{n:>6}", fmt_bytes(io_std), f"{t_std:.2f} ms",
                 fmt_bytes(io_fa), f"{t_fa:.2f} ms",
                 f"{io_std/io_fa:.1f}×"])
print_table(['n', 'Std IO', 'Std Time', 'FA IO', 'FA Time', 'Speedup'], rows)

## §5 PagedAttention

In [ ]:
# === §5: PagedAttention ===
np.random.seed(42)

def static_allocation_waste(n_requests, max_length, actual_lengths, kv_bytes_per_token):
    """Compute memory waste for static KV cache allocation.
    
    Returns:
        allocated, used, waste, utilisation
    """
    allocated = n_requests * max_length * kv_bytes_per_token
    used = sum(actual_lengths) * kv_bytes_per_token
    waste = allocated - used
    utilisation = used / allocated
    return allocated, used, waste, utilisation

def paged_allocation_waste(actual_lengths, block_size, kv_bytes_per_token):
    """Compute memory waste for PagedAttention.
    
    Returns:
        allocated, used, waste, utilisation
    """
    total_blocks = sum(int(np.ceil(l / block_size)) for l in actual_lengths)
    allocated = total_blocks * block_size * kv_bytes_per_token
    used = sum(actual_lengths) * kv_bytes_per_token
    waste = allocated - used
    utilisation = used / allocated
    return allocated, used, waste, utilisation

# --- 5a: Fragmentation comparison ---
print("=== §5: PagedAttention ===\n")

n_requests = 8
max_length = 2048
# Simulate variable actual lengths
actual_lengths = np.array([128, 256, 512, 1024, 64, 768, 384, 192])

# KV per token for LLaMA-3 8B at BF16
kv_per_token = 2 * 32 * 8 * 128 * 2  # 2*L*H_kv*d_k*bytes = 131072 bytes

alloc_s, used_s, waste_s, util_s = static_allocation_waste(
    n_requests, max_length, actual_lengths, kv_per_token)

print(f"Request lengths: {actual_lengths.tolist()}")
print(f"Max length: {max_length}")
print(f"KV bytes per token: {fmt_bytes(kv_per_token)}")

print(f"\nStatic Allocation:")
print(f"  Allocated: {fmt_bytes(alloc_s)}")
print(f"  Used:      {fmt_bytes(used_s)}")
print(f"  Wasted:    {fmt_bytes(waste_s)}")
print(f"  Utilisation: {util_s:.1%}")

print("\nPagedAttention:")
for block_size in [4, 16, 64, 256]:
    alloc_p, used_p, waste_p, util_p = paged_allocation_waste(
        actual_lengths, block_size, kv_per_token)
    print(f"  Block size {block_size:>3}: allocated {fmt_bytes(alloc_p):>10}, "
          f"waste {fmt_bytes(waste_p):>8}, util {util_p:.1%}")

# --- 5b: Concurrent request capacity ---
print("\n--- Concurrent Request Capacity (80 GB GPU, 16 GB for weights) ---")
available_mem = 64e9  # 80 - 16 for weights

rows = []
for avg_len in [256, 512, 1024, 2048, 4096]:
    # Static: allocate max_length=4096 per request
    static_per_req = 4096 * kv_per_token
    static_reqs = int(available_mem / static_per_req)
    
    # Paged: allocate actual length
    paged_per_req = int(np.ceil(avg_len / 16)) * 16 * kv_per_token
    paged_reqs = int(available_mem / paged_per_req)
    
    improvement = paged_reqs / max(static_reqs, 1)
    rows.append([f"{avg_len}", f"{static_reqs}", f"{paged_reqs}", f"{improvement:.1f}×"])

print_table(['Avg Length', 'Static', 'Paged', 'Improvement'], rows)

## §6 Speculative Decoding

In [ ]:
# === §6: Speculative Decoding ===
np.random.seed(42)

def expected_tokens_accepted(alpha, K):
    """Expected tokens from K draft tokens with acceptance rate alpha.
    
    Formula: (1 - alpha^(K+1)) / (1 - alpha)
    """
    if alpha >= 1.0:
        return K + 1
    return (1 - alpha**(K+1)) / (1 - alpha)

def speculative_speedup(alpha, K, draft_cost_ratio):
    """Compute speedup from speculative decoding.
    
    Args:
        alpha: expected per-token acceptance rate
        K: number of draft tokens
        draft_cost_ratio: C_draft / C_target
    Returns:
        speedup vs standard autoregressive decoding
    """
    E_tokens = expected_tokens_accepted(alpha, K)
    # Standard: E_tokens individual target forward passes
    standard_cost = E_tokens  # normalised to target cost = 1
    # Speculative: K draft passes + 1 target pass
    spec_cost = K * draft_cost_ratio + 1
    return standard_cost / spec_cost

def simulate_speculative_decoding(p_draft, p_target, K, n_steps=1000):
    """Simulate speculative decoding and measure actual acceptance.
    
    Args:
        p_draft: draft model probability distribution (array of vocab_size)
        p_target: target model probability distribution
        K: draft tokens per step
        n_steps: number of speculation rounds
    Returns:
        total_tokens, total_target_calls, avg_acceptance
    """
    total_tokens = 0
    total_target_calls = 0
    
    for _ in range(n_steps):
        accepted = 0
        for k in range(K):
            # Draft proposes token x
            x = np.random.choice(len(p_draft), p=p_draft)
            # Accept with min(1, q(x)/p(x))
            accept_prob = min(1.0, p_target[x] / p_draft[x])
            if np.random.random() < accept_prob:
                accepted += 1
            else:
                break
        # +1 for the resampled/corrected token
        total_tokens += accepted + 1
        total_target_calls += 1
    
    return total_tokens, total_target_calls, total_tokens / total_target_calls

# --- 6a: Expected tokens table ---
print("=== §6: Speculative Decoding ===\n")

print("Expected tokens per target call:")
rows = []
for alpha in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    row = [f"{alpha:.2f}"]
    for K in [2, 4, 8, 16]:
        E = expected_tokens_accepted(alpha, K)
        row.append(f"{E:.2f}")
    rows.append(row)
print_table(['α', 'K=2', 'K=4', 'K=8', 'K=16'], rows)

# --- 6b: Speedup analysis ---
print("\n--- Speedup Analysis (draft = 10% of target cost) ---")
draft_ratio = 0.10  # draft is 10% of target cost
rows = []
for alpha in [0.5, 0.7, 0.8, 0.9, 0.95]:
    row = [f"{alpha:.2f}"]
    for K in [2, 4, 8]:
        s = speculative_speedup(alpha, K, draft_ratio)
        row.append(f"{s:.2f}×")
    rows.append(row)
print_table(['α', 'K=2', 'K=4', 'K=8'], rows)

# --- 6c: Simulation ---
print("\n--- Simulation: Speculative Decoding ---")
vocab_size = 100

# Target: peaked distribution
logits_target = np.random.randn(vocab_size) * 2
p_target = np.exp(logits_target) / np.sum(np.exp(logits_target))

# Draft: similar but noisier
noise = np.random.randn(vocab_size) * 0.5
logits_draft = logits_target + noise
p_draft = np.exp(logits_draft) / np.sum(np.exp(logits_draft))

for K in [2, 4, 8]:
    tokens, calls, avg = simulate_speculative_decoding(p_draft, p_target, K, n_steps=5000)
    print(f"  K={K:>2}: avg {avg:.2f} tokens/call "
          f"(theoretical with perfect α: {expected_tokens_accepted(0.8, K):.2f})")

# --- 6d: Acceptance probability proof verification ---
print("\n--- Verifying Output Distribution is Exact ---")
# Run many single-token speculations and check empirical distribution = p_target
n_samples = 50000
K_test = 1
empirical_counts = np.zeros(vocab_size)

for _ in range(n_samples):
    x = np.random.choice(vocab_size, p=p_draft)
    accept_prob = min(1.0, p_target[x] / p_draft[x])
    if np.random.random() < accept_prob:
        empirical_counts[x] += 1
    else:
        # Resample from max(0, q - p) normalised
        residual = np.maximum(0, p_target - p_draft)
        if np.sum(residual) > 0:
            residual /= np.sum(residual)
            x_new = np.random.choice(vocab_size, p=residual)
            empirical_counts[x_new] += 1
        else:
            empirical_counts[x] += 1

empirical_dist = empirical_counts / np.sum(empirical_counts)
kl_div = np.sum(p_target * np.log(p_target / (empirical_dist + 1e-10) + 1e-10))
max_diff = np.max(np.abs(empirical_dist - p_target))
print(f"  KL(target || empirical) = {kl_div:.6f} (should be ≈ 0)")
print(f"  Max |p_target - p_empirical| = {max_diff:.4f}")
print(f"  Output distribution matches target: {max_diff < 0.02}")

## §7 Quantisation Error

In [ ]:
# === §7: Quantisation Error ===
np.random.seed(42)

def quantise(x, bits, mode='symmetric'):
    """Quantise tensor to given bit width.
    
    Args:
        x: input tensor (float)
        bits: number of bits (4 or 8)
        mode: 'symmetric' or 'asymmetric'
    Returns:
        x_dequant: dequantised values, scale, zero_point
    """
    if mode == 'symmetric':
        qmax = 2**(bits-1) - 1
        qmin = -2**(bits-1)
        abs_max = np.max(np.abs(x))
        scale = abs_max / qmax if abs_max > 0 else 1.0
        zero_point = 0
    else:
        qmax = 2**bits - 1
        qmin = 0
        scale = (np.max(x) - np.min(x)) / qmax if np.max(x) != np.min(x) else 1.0
        zero_point = np.min(x)
    
    x_q = np.clip(np.round((x - zero_point) / scale), qmin, qmax).astype(int)
    x_dequant = scale * x_q + zero_point
    return x_dequant, scale, zero_point

def quantise_grouped(x, bits, group_size):
    """Per-group quantisation for better accuracy.
    
    Args:
        x: input tensor (1D)
        bits: number of bits
        group_size: elements per group
    Returns:
        x_dequant: dequantised values, n_scales
    """
    n = len(x)
    x_dequant = np.zeros_like(x)
    n_groups = int(np.ceil(n / group_size))
    
    for g in range(n_groups):
        start = g * group_size
        end = min(start + group_size, n)
        x_dequant[start:end], _, _ = quantise(x[start:end], bits)
    
    return x_dequant, n_groups

# --- 7a: Basic quantisation demo ---
print("=== §7: Quantisation Error ===\n")

# Simulate a weight tensor
weights = np.random.randn(1024) * 0.02  # typical transformer weight scale

rows = []
for bits in [8, 4, 3, 2]:
    w_q, scale, zp = quantise(weights, bits)
    mse = np.mean((weights - w_q)**2)
    max_err = np.max(np.abs(weights - w_q))
    snr = 10 * np.log10(np.mean(weights**2) / mse) if mse > 0 else np.inf
    rows.append([f"INT{bits}", f"{scale:.6f}", f"{mse:.2e}", f"{max_err:.2e}", f"{snr:.1f} dB"])
print_table(['Format', 'Scale', 'MSE', 'Max Error', 'SNR'], rows)

# --- 7b: Outlier impact ---
print("\n--- Outlier Impact on Quantisation ---")
weights_clean = np.random.randn(1024) * 0.02
weights_outlier = weights_clean.copy()
weights_outlier[0] = 2.0  # 100× larger than typical

for desc, w in [('No outliers', weights_clean), ('With outlier (100×)', weights_outlier)]:
    w_q4, s, _ = quantise(w, 4)
    mse = np.mean((w - w_q4)**2)
    print(f"  {desc:30s}: INT4 MSE = {mse:.2e}, scale = {s:.6f}")

print("  → Outlier stretches the scale; all other weights lose precision")

# --- 7c: Per-group quantisation ---
print("\n--- Per-Group Quantisation (INT4) ---")
rows = []
for group_size in [1024, 128, 64, 32]:
    w_gq, n_groups = quantise_grouped(weights_outlier, 4, group_size)
    mse = np.mean((weights_outlier - w_gq)**2)
    overhead = n_groups * 4 / (len(weights_outlier) * 0.5) * 100  # scale overhead vs data
    rows.append([f"{group_size}", f"{n_groups}", f"{mse:.2e}", f"{overhead:.1f}%"])
print_table(['Group Size', 'N Groups', 'MSE', 'Scale Overhead'], rows)
print("Smaller groups → better accuracy but more scale factors to store")

# --- 7d: Model size comparison ---
print("\n--- LLaMA-3 70B Model Size ---")
n_params = 70e9
rows = []
for label, bpp in [('FP32', 4), ('BF16', 2), ('INT8', 1), ('INT4', 0.5)]:
    size = n_params * bpp
    rows.append([label, f"{bpp}", fmt_bytes(size)])
print_table(['Format', 'Bytes/Param', 'Total Size'], rows)

## §8 SmoothQuant

In [ ]:
# === §8: SmoothQuant ===
np.random.seed(42)

def smoothquant_transform(X, W, alpha=0.5):
    """Apply SmoothQuant: migrate quantisation difficulty from activations to weights.
    
    Y = X @ W = (X * diag(s)^-1) @ (diag(s) @ W)
    
    Args:
        X: activations [batch, d_in]
        W: weights [d_in, d_out]
        alpha: smoothing strength (0=all on weights, 1=all on activations)
    Returns:
        X_smooth, W_smooth, scale
    """
    # Per-channel max of activations and weights
    act_max = np.max(np.abs(X), axis=0)  # [d_in]
    w_max = np.max(np.abs(W), axis=1)    # [d_in]
    
    # Smoothing scale: s_j = max(|X_j|)^alpha / max(|W_j|)^(1-alpha)
    scale = (act_max ** alpha) / (w_max ** (1 - alpha) + 1e-10)
    scale = np.maximum(scale, 1e-5)  # prevent division by zero
    
    # Transform
    X_smooth = X / scale[np.newaxis, :]     # divide activations by s
    W_smooth = W * scale[:, np.newaxis]      # multiply weights by s
    
    return X_smooth, W_smooth, scale

# --- 8a: Demonstrate smoothing ---
print("=== §8: SmoothQuant ===\n")

d_in, d_out, batch = 64, 32, 16

# Activations with outlier channels (simulating transformer behaviour)
X = np.random.randn(batch, d_in) * 0.5
outlier_channels = [5, 12, 33, 47]  # these channels have huge values
for ch in outlier_channels:
    X[:, ch] *= 50  # 50× larger

# Weights: relatively uniform
W = np.random.randn(d_in, d_out) * 0.02

# Ground truth output
Y_true = X @ W

# --- 8b: Quantise without SmoothQuant ---
print("--- Without SmoothQuant (INT8) ---")
# Quantise activations and weights separately per-tensor
X_q, _, _ = quantise(X.flatten(), 8)
X_q = X_q.reshape(X.shape)
W_q, _, _ = quantise(W.flatten(), 8)
W_q = W_q.reshape(W.shape)
Y_naive = X_q @ W_q
mse_naive = np.mean((Y_true - Y_naive)**2)
print(f"  Activation range: [{X.min():.2f}, {X.max():.2f}]")
print(f"  Weight range:     [{W.min():.4f}, {W.max():.4f}]")
print(f"  MSE(Y_true, Y_quant): {mse_naive:.6f}")

# --- 8c: Quantise with SmoothQuant ---
print("\n--- With SmoothQuant (INT8, alpha=0.5) ---")
X_sm, W_sm, scale = smoothquant_transform(X, W, alpha=0.5)

X_sm_q, _, _ = quantise(X_sm.flatten(), 8)
X_sm_q = X_sm_q.reshape(X_sm.shape)
W_sm_q, _, _ = quantise(W_sm.flatten(), 8)
W_sm_q = W_sm_q.reshape(W_sm.shape)
Y_smooth = X_sm_q @ W_sm_q
mse_smooth = np.mean((Y_true - Y_smooth)**2)

print(f"  Smoothed activation range: [{X_sm.min():.2f}, {X_sm.max():.2f}]")
print(f"  Smoothed weight range:     [{W_sm.min():.4f}, {W_sm.max():.4f}]")
print(f"  MSE(Y_true, Y_smooth):     {mse_smooth:.6f}")
print(f"  Improvement: {mse_naive/mse_smooth:.1f}× lower error")

# --- 8d: Alpha sweep ---
print("\n--- Alpha Sweep ---")
rows = []
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    X_s, W_s, _ = smoothquant_transform(X, W, alpha=alpha)
    X_sq, _, _ = quantise(X_s.flatten(), 8)
    X_sq = X_sq.reshape(X_s.shape)
    W_sq, _, _ = quantise(W_s.flatten(), 8)
    W_sq = W_sq.reshape(W_s.shape)
    Y_sq = X_sq @ W_sq
    mse = np.mean((Y_true - Y_sq)**2)
    act_range = np.max(np.abs(X_s))
    w_range = np.max(np.abs(W_s))
    rows.append([f"{alpha:.2f}", f"{act_range:.2f}", f"{w_range:.4f}", f"{mse:.6f}"])
print_table(['Alpha', 'Act Range', 'W Range', 'MSE'], rows)
print("alpha=0.5 typically best: balanced difficulty between activations and weights")

## §9 Batch Size vs Arithmetic Intensity

In [ ]:
# === §9: Batch Size vs Arithmetic Intensity ===
np.random.seed(42)

def decode_arithmetic_intensity(batch_size, seq_len, d_model, bytes_per_elem=2):
    """Arithmetic intensity for decode step (linear layer).
    
    Compute: Y = X @ W where X is [B, d], W is [d, d]
    FLOPs: 2 * B * d * d
    Bytes: W (d*d*bpe) + X (B*d*bpe) + Y (B*d*bpe)
    """
    d = d_model
    flops = 2 * batch_size * d * d
    bytes_accessed = (d * d + 2 * batch_size * d) * bytes_per_elem
    return flops / bytes_accessed

def effective_throughput(batch_size, d_model, peak_flops, peak_bw, bytes_per_elem=2):
    """Effective token throughput at given batch size."""
    I = decode_arithmetic_intensity(batch_size, 0, d_model, bytes_per_elem)
    achieved_flops = roofline_throughput(I, peak_flops, peak_bw)
    # Tokens/sec = achieved_flops / (flops_per_token)
    flops_per_token = 2 * d_model * d_model  # per-layer, per token
    return achieved_flops / flops_per_token

# --- 9a: Intensity vs batch size ---
print("=== §9: Batch Size vs Arithmetic Intensity ===\n")

d_model = 4096  # typical hidden dim

print(f"Linear layer Y = X @ W (d={d_model}, BF16):")
rows = []
for B in [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]:
    I = decode_arithmetic_intensity(B, 0, d_model)
    regime_a100 = 'BANDWIDTH' if I < 156 else 'COMPUTE'
    regime_h100 = 'BANDWIDTH' if I < 295 else 'COMPUTE'
    rows.append([f"{B:>4}", f"{I:.1f}", regime_a100, regime_h100])
print_table(['Batch', 'Intensity', 'A100 (I*=156)', 'H100 (I*=295)'], rows)

# --- 9b: Throughput vs batch size ---
print("\n--- Decode Throughput vs Batch Size ---")
a100 = {'flops': 312e12, 'bw': 2.0e12}
h100 = {'flops': 989e12, 'bw': 3.35e12}

rows = []
for B in [1, 4, 16, 64, 128, 256, 512]:
    thr_a100 = effective_throughput(B, d_model, a100['flops'], a100['bw'])
    thr_h100 = effective_throughput(B, d_model, h100['flops'], h100['bw'])
    rows.append([f"{B:>4}", f"{thr_a100:.0f}", f"{thr_h100:.0f}",
                 f"{thr_a100/B:.1f}", f"{thr_h100/B:.1f}"])
print_table(['Batch', 'A100 tok/s', 'H100 tok/s', 'A100 /user', 'H100 /user'], rows)

# --- 9c: Optimal batch size ---
print("\n--- Optimal Batch Size (A100, d=4096) ---")
batches = np.arange(1, 513)
intensities = [decode_arithmetic_intensity(B, 0, d_model) for B in batches]
ridge_a100 = 156

optimal_B = None
for B, I in zip(batches, intensities):
    if I >= ridge_a100:
        optimal_B = B
        break

print(f"  Ridge point I* = {ridge_a100} FLOPs/byte")
print(f"  Optimal batch size (A100): B ≥ {optimal_B}")
print(f"  Below B={optimal_B}: bandwidth-bound (increasing B helps ~linearly)")
print(f"  Above B={optimal_B}: compute-bound (increasing B helps via amortisation)")

# --- 9d: Quantisation shifts the curve ---
print("\n--- Quantisation Shifts Intensity ---")
B_test = 64
for label, bpe in [('BF16', 2), ('INT8', 1), ('INT4', 0.5)]:
    I = decode_arithmetic_intensity(B_test, 0, d_model, bpe)
    regime = 'COMPUTE' if I >= ridge_a100 else 'BANDWIDTH'
    print(f"  B={B_test}, {label:5s}: I = {I:.1f} ({regime})")
print("  → INT4 doubles arithmetic intensity vs INT8; quadruples vs BF16")

## §10 Decode Speed Limit

In [ ]:
# === §10: Decode Speed Limit ===
np.random.seed(42)

def max_decode_tokens_per_sec(model_bytes, kv_bytes_per_step, bandwidth, batch_size=1):
    """Theoretical maximum decode speed (bandwidth-limited).
    
    Each token requires loading all model weights + KV cache update.
    At batch=B, weights amortised over B tokens.
    
    Args:
        model_bytes: total model weight size in bytes
        kv_bytes_per_step: KV cache bytes read per token per request
        bandwidth: HBM bandwidth in bytes/sec
        batch_size: number of concurrent requests
    Returns:
        tokens/sec total, tokens/sec per request
    """
    # Per decode step: read weights (shared) + KV cache for each request
    bytes_per_step = model_bytes + batch_size * kv_bytes_per_step
    steps_per_sec = bandwidth / bytes_per_step
    total_tok_sec = steps_per_sec * batch_size
    per_user = steps_per_sec
    return total_tok_sec, per_user

def model_size_bytes(n_params, bits):
    """Model size in bytes."""
    return n_params * bits / 8

# --- 10a: Decode speed limits ---
print("=== §10: Decode Speed Limit ===\n")

# Model configs
model_configs = [
    ('LLaMA-3 8B',  8e9,  32, 8, 128),
    ('LLaMA-3 70B', 70e9, 80, 8, 128),
]

hw_configs = [
    ('A100', 2.0e12),
    ('H100', 3.35e12),
    ('H200', 4.8e12),
]

print("Max decode speed (batch=1, s=2048, BF16):")
rows = []
for model_name, n_params, L, H_kv, d_k in model_configs:
    m_bytes = model_size_bytes(n_params, 16)  # BF16
    kv_per_step = 2 * L * H_kv * d_k * 2 * 2048  # approximate KV bytes (simplified)
    for hw_name, bw in hw_configs:
        total, per_user = max_decode_tokens_per_sec(m_bytes, 0, bw, batch_size=1)
        rows.append([model_name, hw_name, fmt_bytes(m_bytes), f"{per_user:.0f} tok/s"])
print_table(['Model', 'GPU', 'Size', 'Max tok/s (B=1)'], rows)

# --- 10b: Quantisation impact on speed ---
print("\n--- Quantisation Impact on Decode Speed (H100, batch=1) ---")
bw_h100 = 3.35e12
rows = []
for model_name, n_params, _, _, _ in model_configs:
    for label, bits in [('BF16', 16), ('INT8', 8), ('INT4', 4)]:
        m_bytes = model_size_bytes(n_params, bits)
        total, per_user = max_decode_tokens_per_sec(m_bytes, 0, bw_h100)
        rows.append([model_name, label, fmt_bytes(m_bytes), f"{per_user:.0f} tok/s"])
print_table(['Model', 'Format', 'Size', 'Max tok/s'], rows)

# --- 10c: Batch size vs throughput vs latency ---
print("\n--- Batch Size vs Throughput (LLaMA-3 70B, BF16, H100) ---")
m_bytes_70b = model_size_bytes(70e9, 16)
kv_per_step_70b = 2 * 80 * 8 * 128 * 2  # per token per layer per request

rows = []
for B in [1, 2, 4, 8, 16, 32, 64]:
    total, per_user = max_decode_tokens_per_sec(m_bytes_70b, kv_per_step_70b, bw_h100, B)
    latency = 1000 / per_user  # ms per token
    rows.append([f"{B:>3}", f"{total:.0f}", f"{per_user:.1f}", f"{latency:.1f} ms"])
print_table(['Batch', 'Total tok/s', 'Per-user tok/s', 'TPOT'], rows)

# --- 10d: Cost analysis ---
print("\n--- Cost per Million Tokens ---")
gpu_costs = {'A100': 2.0, 'H100': 3.0, 'H200': 4.0}  # $/hour spot

rows = []
for model_name, n_params, _, _, _ in model_configs:
    for bits_label, bits in [('BF16', 16), ('INT4', 4)]:
        m_bytes = model_size_bytes(n_params, bits)
        n_gpus = max(1, int(np.ceil(m_bytes / 80e9)))  # GPUs needed
        for hw_name, bw in hw_configs:
            total, _ = max_decode_tokens_per_sec(m_bytes, 0, bw, batch_size=32)
            cost_hour = gpu_costs.get(hw_name, 3.0) * n_gpus
            tokens_per_hour = total * 3600
            cost_per_M = cost_hour / (tokens_per_hour / 1e6) if tokens_per_hour > 0 else np.inf
            rows.append([f"{model_name} {bits_label}", hw_name,
                         f"{n_gpus}", f"{total:.0f}",
                         f"${cost_per_M:.2f}"])
print_table(['Config', 'GPU', 'GPUs', 'tok/s (B=32)', '$/M tok'], rows)

print("\n" + "=" * 60)
print("THEORY NOTEBOOK COMPLETE")
print("=" * 60)
print("\nKey takeaways:")
print("  1. Decode is bandwidth-bound: roofline model proves this")
print("  2. KV cache is the primary memory bottleneck at long contexts")
print("  3. Online softmax enables FlashAttention (exact, no materialise)")
print("  4. Speculative decoding gives exact target distribution, 2-4× speedup")
print("  5. Quantisation reduces both memory and bandwidth requirements")
print("  6. SmoothQuant makes activation quantisation viable")
print("  7. Batch size determines bandwidth vs compute regime")
print("  8. INT4 gives 4× decode speedup on same hardware")